In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector


load_dotenv(override=True)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
    embeddings=embedding_model,
    collection_name="integrated_university_rules",
    connection=connection_url,
    use_jsonb=True, #메타 데이터를 JSONB 형식으로 저장할지 여부를 결정합니다. True로 설정하면 메타데이터가 JSONB 형식으로 저장됩니다. False로 설정하면 일반 텍스트 형식으로 저장됩니다.
)

print("PostgreSQL + pgvector 연결 및 초기화 완료")

In [ ]:
from langchain_core.documents import Document

# 1차 등록 대상 문서 (2026 규정 및 장학 규정)
initial_documents = [
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    )
]


In [ ]:
from langchain_classic.indexes import SQLRecordManager, index

record_manager = SQLRecordManager(
    namespace="university_rules",
    db_url=connection_url 
)

record_manager.create_schema()  # 추적용 테이블 생성

In [ ]:
first_result = index(
    docs_source=initial_documents,
    record_manager=record_manager,
    vector_store=vector_store,
    source_id_key="source",
    cleanup="incremental",
)

print(first_result)

# none : 증분 관리x
# icremental : 증분 관리o / 중복된 레코드가 있으면 삭제하고 새로 추가
# full : 증분 관리x / 기존 데이터 삭제 후 새로 추가

In [ ]:
updated_documents = [
    Document(
        page_content="[개정 도서관 규정] 학부생은 최대 5권까지 14일간 대출 가능합니다.",
        metadata={"source": "rules_library.txt", "category": "library", "year": 2026}
    ),
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상 시 장학금 대상입니다.",
        metadata={"source": "rules_scholarship.txt", "category": "scholarship", "year": 2026}
    )
]

second_result = index(
    docs_source=updated_documents,
    record_manager=record_manager,
    vector_store=vector_store,
    source_id_key="source",
    cleanup="incremental",
)

print(second_result)